# Week 2–3 · Practice exercises: SOLUTIONS

Try `practice_exercises.ipynb` first! Every check below passes.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
___ = None          # placeholder so unanswered cells fail gracefully

def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)

## 1 · Parameters of a conv layer (nb 01)
A conv layer has `out_channels` kernels, each of size `kernel × kernel × in_channels`, plus one bias per kernel. Write the formula for `Conv2d(3, 64, kernel_size=3)`.

In [2]:
n = 64 * (3 * 3 * 3) + 64

check('conv parameters', n == sum(p.numel() for p in nn.Conv2d(3, 64, 3).parameters()))

PASS  conv parameters


## 2 · Output size (nb 02, LeNet)
Use output = (N − K + 2P)/S + 1. A 32×32 input goes through conv 5×5 (no padding, stride 1), then 2×2 max-pool (stride 2), then conv 5×5, then 2×2 max-pool. What is the final height/width?

In [3]:
after_conv1 = (32 - 5) // 1 + 1      # 28
after_pool1 = after_conv1 // 2       # 14
after_conv2 = after_pool1 - 5 + 1     # 10
after_pool2 = after_conv2 // 2        # 5

x = torch.zeros(1, 1, 32, 32)
x = F.max_pool2d(nn.Conv2d(6, 16, 5)(F.max_pool2d(nn.Conv2d(1, 6, 5)(x), 2)), 2)
check('LeNet shapes', (after_conv1, after_pool1, after_conv2, after_pool2) == (28, 14, 10, x.shape[-1]))

PASS  LeNet shapes


## 3 · ReLU by hand (nb 01)
Implement ReLU without `torch.relu`: keep positives, turn negatives into 0. Hint: `torch.clamp(x, min=...)` or `torch.maximum`.

In [4]:
def my_relu(x):
    return torch.clamp(x, min=0)

t = torch.tensor([-3., -0.5, 0., 2., 7.])
check('ReLU', torch.equal(my_relu(t), torch.relu(t)))

PASS  ReLU


## 4 · Dropout arithmetic (nb 01)
A layer has 40 neurons and dropout p = 0.25. (a) How many neurons are dropped per training pass on average? (b) By what factor are the surviving outputs scaled in PyTorch?

In [5]:
dropped_on_average = 40 * 0.25
scale_factor = 1 / (1 - 0.25)

d = nn.Dropout(0.25); d.train(); out = d(torch.ones(1000))
check('dropped count', dropped_on_average == 10)
check('scale factor', abs(scale_factor - out.max().item()) < 1e-5)

PASS  dropped count
PASS  scale factor


## 5 · Stacked 3×3 receptive field (nb 02, VGG)
How big a window (n × n) do 4 stacked 3×3 convolutions (stride 1) see? And how many weights do they need for C = 32 channels in and out (no bias), compared with one 9×9 layer?

In [6]:
receptive_field = 1 + 4 * 2          # each 3x3 adds 2 -> 9
weights_stacked = 4 * 3 * 3 * 32 * 32
weights_single_9x9 = 9 * 9 * 32 * 32

check('receptive field', receptive_field == 9)
check('stacked is cheaper', weights_stacked == 36864 and weights_single_9x9 == 82944)

PASS  receptive field
PASS  stacked is cheaper


## 6 · A residual block (nb 03, ResNet)
Complete `forward` so the block returns ReLU(F(x) + x), where F(x) = conv2(relu(conv1(x))).

In [7]:
class Residual(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1)
    def forward(self, x):
        return F.relu(self.conv2(F.relu(self.conv1(x))) + x)

blk = Residual(4)
with torch.no_grad():
    for p in blk.parameters(): p.zero_()          # F(x) = 0 -> block must pass x through
x = torch.rand(1, 4, 8, 8)
check('identity when F=0', torch.allclose(blk(x), x))

PASS  identity when F=0


## 7 · Compound scaling (nb 03, EfficientNet)
With α = 1.2, β = 1.1, γ = 1.15 and φ = 3, compute the depth multiplier, the width multiplier and the input resolution (base 224, rounded to a whole number).

In [8]:
depth_mult = 1.2 ** 3
width_mult = 1.1 ** 3
resolution = round(224 * 1.15 ** 3)

check('depth', abs(depth_mult - 1.728) < 1e-3)
check('width', abs(width_mult - 1.331) < 1e-3)
check('resolution', resolution == 341)

PASS  depth
PASS  width
PASS  resolution


## 8 · IoU (nb 04)
Write IoU for boxes given as (x1, y1, x2, y2).

In [9]:
def iou(a, b):
    inter_w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union

from torchvision.ops import box_iou
A, B = (0, 0, 10, 10), (5, 5, 15, 15)
check('IoU overlap', abs(iou(A, B) - box_iou(torch.tensor([A], dtype=torch.float), torch.tensor([B], dtype=torch.float)).item()) < 1e-6)
check('IoU no overlap', iou((0, 0, 1, 1), (5, 5, 6, 6)) == 0)

PASS  IoU overlap
PASS  IoU no overlap


## 9 · Precision and recall (nb 04)
A detector drew 50 boxes; 40 were correct. The image set contained 80 real objects. Compute precision and recall.

In [10]:
precision = 40 / 50     # of what we predicted, how much was right
recall = 40 / 80        # of what exists, how much we found

check('precision', precision == 0.8)
check('recall', recall == 0.5)

PASS  precision
PASS  recall


## 10 · U-Net shapes (nb 05)
A decoder upsamples a (1, 64, 8, 8) map with `ConvTranspose2d(64, 32, kernel_size=2, stride=2)` and concatenates the skip feature map of shape (1, 32, 16, 16). What is the shape after concatenation?

In [11]:
shape_after_concat = (1, 64, 16, 16)    # 32 upsampled channels + 32 skip channels

up = nn.ConvTranspose2d(64, 32, 2, stride=2)(torch.zeros(1, 64, 8, 8))
check('concat shape', tuple(torch.cat([up, torch.zeros(1, 32, 16, 16)], 1).shape) == shape_after_concat)

PASS  concat shape


## 11 · LSTM parameter count (nb 06)
An LSTM has 4 gate-like parts (input, forget, candidate, output). Each has weights for the input (hidden × input), weights for the previous hidden state (hidden × hidden) and two bias vectors in PyTorch. Count the parameters of `nn.LSTM(10, 20)`.

In [12]:
n_lstm = 4 * (20 * 10 + 20 * 20 + 20 + 20)

check('LSTM parameters', n_lstm == sum(p.numel() for p in nn.LSTM(10, 20).parameters()))

PASS  LSTM parameters


## 12 · Scaled dot-product attention (nb 07)
Implement softmax(Q·Kᵀ / √d) · V. `d` is the last dimension of Q.

In [13]:
def attention(Q, K, V):
    d = Q.shape[-1]
    weights = torch.softmax(Q @ K.transpose(-2, -1) / d ** 0.5, dim=-1)
    return weights @ V

torch.manual_seed(0); Q, K, V = torch.randn(5, 8), torch.randn(5, 8), torch.randn(5, 8)
check('attention', torch.allclose(attention(Q, K, V), F.scaled_dot_product_attention(Q[None], K[None], V[None])[0], atol=1e-5))

PASS  attention


## 13 · ViT token count (nb 07)
A ViT takes 224×224 images with 16×16 patches and adds one [CLS] token. How many tokens enter the transformer, and how many attention pairs (tokens × tokens) does one head compute?

In [14]:
n_tokens = (224 // 16) ** 2 + 1
n_pairs = n_tokens ** 2

check('tokens', n_tokens == 197)
check('pairs', n_pairs == 38809)

PASS  tokens
PASS  pairs
